In [2]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Device count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}: {torch.cuda.get_device_name(i)}")


PyTorch version: 2.10.0+cu130
CUDA available: True
CUDA version: 13.0
Device count: 2
Device 0: NVIDIA GeForce RTX 3060 Ti
Device 1: NVIDIA GeForce RTX 3060 Ti


In [3]:
from huggingface_hub import hf_hub_download
import numpy as np
from aeon.transformations.collection.convolution_based import Rocket, MiniRocket, MultiRocket, HydraTransformer
from aeon.transformations.collection.feature_based import Catch22
from aeon.transformations.collection.interval_based import QUANTTransformer
from sklearn.metrics import accuracy_score

In [ ]:
path_X = hf_hub_download(repo_id = f"monster-monash/WISDM", filename = f"WISDM_X.npy", repo_type = "dataset")
path_y = hf_hub_download(repo_id = f"monster-monash/WISDM", filename = f"WISDM_y.npy", repo_type = "dataset")
X = np.load(path_X, mmap_mode = "r")
y = np.load(path_y, mmap_mode = "r")

In [6]:
len(np.unique(y))

82

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [5]:
#rocket = Rocket(n_jobs=-1, random_state=0)
#Xtr_rocket = rocket.fit_transform(X_train, y_train)
#Xte_rocket = rocket.transform(X_test)

In [6]:
#minirocket = MiniRocket(n_jobs=-1, random_state=0)
#Xtr_mr = minirocket.fit_transform(X_train, y_train)
#Xte_mr = minirocket.transform(X_test)

In [7]:
multirocket = MultiRocket(n_jobs=-1)
Xtr_multir = multirocket.fit_transform(X_train, y_train)
Xte_multir = multirocket.transform(X_test)

In [8]:
#hydra = HydraTransformer(n_jobs=-1)
#Xtr_hydra = hydra.fit_transform(X_train, y_train)
#Xte_hydra = hydra.transform(X_test)

In [9]:
#quant = QUANTTransformer()
#Xtr_quant = np.array(quant.fit_transform(X_train, y_train))
#Xte_quant = np.array(quant.transform(X_test))

In [10]:
#catch22 = Catch22(n_jobs=-1)
#Xtr_c22 = catch22.fit_transform(X_train, y_train)
#Xte_c22 = catch22.transform(X_test)

In [11]:
#print(Xtr_rocket.shape, Xtr_mr.shape, Xtr_multir.shape, Xtr_hydra.shape, Xtr_quant.shape, Xtr_c22.shape)

In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler(with_mean=False)

Xtr_scaled = scaler.fit_transform(Xtr_multir)
Xte_scaled = scaler.transform(Xte_multir)


In [13]:
#import numpy as np
#from sklearn.linear_model import SGDClassifier
#from sklearn.metrics import accuracy_score
#
#classes = np.unique(y_train)
#
#clf = SGDClassifier(
#    loss="log_loss",
#    penalty="l2",
#    alpha=1e-4,
#    max_iter=1,        # one epoch per call
#    tol=None,          # IMPORTANT
#    n_jobs=10,
#    verbose=0,
#)
#
#n_epochs = 10
#
#for epoch in range(1, n_epochs + 1):
#    clf.partial_fit(Xtr_scaled, y_train, classes=classes)
#
#    y_pred = clf.predict(Xte_scaled)
#    acc = accuracy_score(y_test, y_pred)
#
#    print(f"Epoch {epoch:02d} | test accuracy = {acc:.4f}")
#

In [14]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

In [15]:
print(torch.cuda.is_available())
print(torch.cuda.current_device())

True
0


In [16]:
import os
print(os.environ.get("CUDA_VISIBLE_DEVICES", "not set"))

not set


In [19]:
#"write train and test into a numpy file"
import numpy as np

np.save("X_train.npy", Xtr_scaled)
np.save("y_train.npy", y_train)
np.save("X_test.npy",  Xte_scaled)
np.save("y_test.npy",  y_test)


In [ ]:
# now load the data

In [ ]:
dfgdfg=sfdgsdf

In [ ]:


Xtr = torch.from_numpy(Xtr_scaled).float()
ytr = torch.from_numpy(y_train).long()

Xte = torch.from_numpy(Xte_scaled).float()
yte = torch.from_numpy(y_test).long()

n_features = Xtr.shape[1]
n_classes = len(torch.unique(ytr))

model = nn.Linear(n_features, n_classes)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=1e-2,          # step size
    weight_decay=1e-4 # L2 penalty (≈ alpha)
)

batch_size = 1024

train_loader = DataLoader(
    TensorDataset(Xtr, ytr),
    batch_size=batch_size,
    shuffle=True,
)

epochs = 20

for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0

    for Xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * yb.size(0)

    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch {epoch:02d} | train loss = {avg_loss:.4f}")


In [ ]:
clf = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-4,
    max_iter=20,        # one epoch per call
    #tol=None,          # IMPORTANT
    n_jobs=20,
    verbose=10,
    #random_state=0,
)
clf.fit(Xtr_scaled, y_train)
y_pred = clf.predict(Xte_scaled)
acc = accuracy_score(y_test, y_pred)
print(f"Final test accuracy = {acc:.4f}")

In [ ]:
sdfsdf=sdfsfd

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier

pipe = Pipeline(
    [
        ("scaler", StandardScaler()),  
        ("clf", SGDClassifier(
            loss="log_loss",
            penalty="l2",
            alpha=1e-4,
            max_iter=1000,
            #tol=1e-3,
            n_jobs=10,
            verbose=10,
            random_state=0,
        )),
    ]
)

pipe.fit(Xtr_multir, y_train)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import RidgeClassifierCV

In [ ]:
for k in [100, 200, 500, 1000, 2000, 5000]:
    pipe = Pipeline([
        ("select", SelectKBest(score_func=f_classif, k=k)),
        ("clf", RidgeClassifierCV())
    ])

    pipe.fit(Xtr_multir, y_train)
    y_pred = pipe.predict(Xte_multir)
    acc = accuracy_score(y_test, y_pred)
    print(f"k={k}, accuracy={acc}")

In [ ]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif

# fit on training data only
skb = SelectKBest(score_func=f_classif, k="all")
skb.fit(Xtr_multir, y_train)

# F-scores for each feature
scores = skb.scores_          # shape: (n_features,)

In [ ]:
len(scores)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(np.sort(scores))

In [ ]:
import time
order = np.argsort(scores)[::-1]

ks = [10, 50, 100, 200, 500, 1000, 2000, 5000]

for k in ks:
    idx = order[:k]

    Xtr_k = Xtr_multir[:, idx]
    Xte_k = Xte_multir[:, idx]

    clf = RidgeClassifierCV(alphas=[0.1, 1.0, 10.0])

    t0 = time.time()
    clf.fit(Xtr_k, y_train)
    train_time = time.time() - t0

    y_pred = clf.predict(Xte_k)
    acc = accuracy_score(y_test, y_pred)

    print(f"k={k:5d} | train_time={train_time:.3f}s | accuracy={acc:.4f}")

In [ ]:
import time

rng = np.random.default_rng(0)

fractions = [0.01, 0.05, 0.10, 0.20, 0.30, 0.50, 0.75, 1.00]

n_train = Xtr_multir.shape[0]

for frac in fractions:
    n_keep = int(frac * n_train)

    idx = rng.choice(n_train, size=n_keep, replace=False)

    Xtr_sub = Xtr_multir[idx]
    ytr_sub = y_train[idx]

    clf = RidgeClassifierCV(alphas=[0.1, 1.0, 10.0])

    t0 = time.time()
    print(Xtr_sub.shape)
    clf.fit(Xtr_sub, ytr_sub)
    train_time = time.time() - t0

    y_pred = clf.predict(Xte_multir)
    acc = accuracy_score(y_test, y_pred)

    print(f"samples={frac:4.0%} ({n_keep:5d}) | train_time={train_time:.3f}s | accuracy={acc:.4f}")
